# Summary_Day17_offline.ipynb  
## 자연어처리 NLP 종합실습 · 인터넷 불가 버전 · 직접 만든 문장 데이터

이 파일은 **인터넷이 안 되는 환경**에서 17강 NLP 흐름을 연습하기 위한 버전이다.

원본 강의는 Hugging Face IMDb dataset, DistilBERT model, tokenizer 다운로드를 사용한다.  
인터넷이 없으면 다운로드가 실패할 수 있다.

그래서 offline 버전은 다음 방식으로 구성한다.

```text
직접 만든 영화 리뷰 문장 데이터
→ 간단 tokenizer
→ vocab 생성
→ padding
→ BasicLSTM / BiLSTM
→ 학습/평가
→ 과제 빈칸 패턴 정리
```

> 주의:  
> offline 버전은 진짜 IMDb 성능을 보는 파일이 아니다.  
> 인터넷 없이 NLP 전처리와 LSTM 구조를 손으로 익히는 대체 실습이다.

## 1. 라이브러리 준비

인터넷을 사용하지 않으므로 `datasets`, `transformers` 없이 PyTorch만 사용한다.

In [ ]:
import re
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(2025)
np.random.seed(2025)
random.seed(2025)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

device = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", device)

## 2. 작은 감성 분류 데이터 만들기

긍정 문장과 부정 문장을 직접 만든다.

```text
label 1 = positive
label 0 = negative
```

In [ ]:
positive_texts = [
    "this movie was wonderful and touching",
    "i loved the acting and the story",
    "the film was great and very enjoyable",
    "what a fantastic and beautiful movie",
    "the characters were lovely and warm",
    "i really liked this excellent film",
    "the story was inspiring and emotional",
    "this was a good and fun experience"
]

negative_texts = [
    "this movie was terrible and boring",
    "i hated the acting and the story",
    "the film was bad and very disappointing",
    "what a awful and painful movie",
    "the characters were weak and cold",
    "i really disliked this poor film",
    "the story was confusing and dull",
    "this was a bad and boring experience"
]

data = [(text, 1) for text in positive_texts] + [(text, 0) for text in negative_texts]

random.shuffle(data)

for text, label in data[:5]:
    print(label, text)

## 3. 토큰화 함수 만들기

정규식으로 영어 단어를 추출한다.

In [ ]:
token_pattern = re.compile(r"[a-z0-9']+")

def simple_tokenize(text):
    return token_pattern.findall(text.lower())

sample = data[0][0]

print("원문:", sample)
print("토큰:", simple_tokenize(sample))

## 4. vocab 만들기

직접 만든 문장 전체에서 token 빈도를 세고 `stoi`, `itos`를 만든다.

In [ ]:
PAD = "<pad>"
UNK = "<unk>"

counter = Counter()

for text, label in data:
    counter.update(simple_tokenize(text))

itos = [PAD, UNK] + [token for token, count in counter.most_common()]
stoi = {token: idx for idx, token in enumerate(itos)}

PAD_IDX = stoi[PAD]
UNK_IDX = stoi[UNK]

print("vocab size:", len(itos))
print("itos:", itos)
print("stoi:", stoi)

## 5. encode 함수

문장을 token id sequence로 바꾸고 길이를 맞춘다.

In [ ]:
MAX_LEN = 10

def encode(text):
    tokens = simple_tokenize(text)
    ids = [stoi.get(token, UNK_IDX) for token in tokens][:MAX_LEN]

    if len(ids) < MAX_LEN:
        ids += [PAD_IDX] * (MAX_LEN - len(ids))

    return ids

print("encoded:", encode(sample))
print("length:", len(encode(sample)))

## 6. Dataset과 DataLoader 만들기

직접 만든 문장 데이터를 PyTorch Dataset으로 감싼다.

In [ ]:
class TinyReviewDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, label = self.data[idx]

        x = torch.tensor(encode(text), dtype=torch.long)
        y = torch.tensor(label, dtype=torch.long)

        return x, y

dataset = TinyReviewDataset(data)

loader = DataLoader(dataset, batch_size=4, shuffle=True)

X_batch, y_batch = next(iter(loader))

print("X_batch:", X_batch.shape)
print("y_batch:", y_batch.shape)
print(X_batch)
print(y_batch)

## 7. BasicLSTM 모델

과제 빈칸에서 나오는 기본 LSTM 구조다.

```text
Embedding
→ LSTM
→ 마지막 hidden hn[0]
→ Linear
```

In [ ]:
class BasicLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)

        self.lstm = nn.LSTM(
            input_size=embed_size,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=False
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        emb = self.embedding(x)

        lstm_out, (hn, cn) = self.lstm(emb)

        logits = self.fc(hn[0])

        return logits

basic_model = BasicLSTM(len(itos), embed_size=32, hidden_size=64, num_classes=2).to(device)

out = basic_model(X_batch.to(device))

print(basic_model)
print("output shape:", out.shape)

## 8. BiLSTM 모델

양방향 LSTM은 forward 방향과 backward 방향 hidden state를 연결한다.

```text
hn[0] = forward
hn[1] = backward
torch.cat((hn[0], hn[1]), dim=1)
```

In [ ]:
class TinyBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size=32, hidden_size=64, num_classes=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=PAD_IDX)

        self.lstm = nn.LSTM(
            input_size=embed_size,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        emb = self.embedding(x)

        lstm_out, (hn, cn) = self.lstm(emb)

        hn_fwd = hn[0]
        hn_bwd = hn[1]

        hidden = torch.cat((hn_fwd, hn_bwd), dim=1)
        hidden = self.dropout(hidden)

        logits = self.fc(hidden)

        return logits

model = TinyBiLSTM(len(itos)).to(device)

out = model(X_batch.to(device))

print(model)
print("output shape:", out.shape)

## 9. 학습/평가 함수

작은 데이터지만 학습 루프는 실제 NLP 모델과 같다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total = 0

    for X, y in loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(X)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)

    return total_loss / total, total_correct / total

history = []

for epoch in range(20):
    loss, acc = train_one_epoch(model, loader, criterion, optimizer, device)
    history.append([epoch + 1, loss, acc])

    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch + 1} | loss={loss:.4f} | acc={acc:.4f}")

history = np.array(history)

In [ ]:
plt.plot(history[:, 0], history[:, 1], label="loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Tiny BiLSTM Loss")
plt.legend()
plt.show()

plt.plot(history[:, 0], history[:, 2], label="accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Tiny BiLSTM Accuracy")
plt.legend()
plt.show()

## 10. 새 문장 추론

학습한 모델에 새 문장을 넣어 긍정/부정 확률을 본다.

In [ ]:
def predict_sentiment(text):
    model.eval()

    x = torch.tensor([encode(text)], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    return probs

test_texts = [
    "this movie was wonderful and fun",
    "this film was boring and terrible"
]

for text in test_texts:
    probs = predict_sentiment(text)
    print(text)
    print(f"negative={probs[0]:.3f}, positive={probs[1]:.3f}")

## 11. DistilBERT 없이 개념만 정리

인터넷이 없으면 DistilBERT 모델과 tokenizer를 다운로드할 수 없다.

그래도 구조는 기억해야 한다.

```text
AutoTokenizer.from_pretrained(model_name)
→ tokenizer(text)
→ DataCollatorWithPadding
→ AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
→ Trainer
→ trainer.train()
→ trainer.evaluate()
```

DistilBERT는 BERT를 경량화한 모델이고, Attention 기반 문맥 이해가 가능하다.

## 12. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `tokenize` | 문장을 token으로 나눔 | 정규식 또는 tokenizer |
| `vocab` | 어휘사전 | token-id 매핑 |
| `stoi` | string to index | token → id |
| `itos` | index to string | id → token |
| `PAD` | padding token | 길이 맞춤 |
| `UNK` | unknown token | 모르는 단어 |
| `Embedding` | token id를 벡터로 변환 | `nn.Embedding` |
| `LSTM` | sequence 모델 | 순서 학습 |
| `BiLSTM` | 양방향 LSTM | 앞뒤 문맥 학습 |
| `hn` | hidden state | 마지막 은닉 상태 |
| `torch.cat` | Tensor 연결 | hidden 결합 |
| `logits` | class 점수 | softmax 전 값 |
| `DistilBERT` | 경량 BERT | 인터넷 필요 |

## 13. 시험용 요약

```text
오프라인 17강 핵심 = 직접 만든 문장 데이터로 NLP 전처리와 LSTM 구조를 연습한다
```

꼭 기억할 것:

- 텍스트는 바로 모델에 넣을 수 없고 token id로 바꿔야 한다.
- tokenization은 문장을 작은 단위로 나누는 것이다.
- vocab은 token과 숫자 id를 연결한다.
- padding은 길이를 맞추는 작업이다.
- embedding은 token id를 dense vector로 바꾼다.
- LSTM은 sequence 순서를 처리한다.
- BiLSTM은 forward/backward hidden을 연결한다.
- DistilBERT는 인터넷 없이는 다운로드가 안 될 수 있다.
- 그래도 DistilBERT 흐름은 tokenizer, collator, model, Trainer 순서로 기억한다.